# FoodGuard AI - Data Preparation

This notebook:
1. **E-Nose Data**
2. **E-Tongue Data**
3. **Vision Deposit Patterns**
4. **Demo Samples**

---

## 1. Imports & Configuration

In [9]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import json
from PIL import Image, ImageDraw
import random
from tqdm import tqdm

sys.path.insert(0, '..')

from foodguard_lib import (
    DB_PATH, ADULTERANTS,
    insert_batch, insert_aroma_analysis, insert_taste_analysis, insert_vision_analysis,
    execute_query, execute_insert, generate_batch_id, generate_vision_images
)

print("[OK] Imports successful")
print(f"Adulterants: {ADULTERANTS}")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Mock prediction function
def mock_predict_class(adulterant):
    """Mock ML classifier - returns predicted class and confidence."""
    return adulterant, 0.95

[OK] Imports successful
Adulterants: ['Fresh', 'Water', 'Urea', 'Detergent', 'Formalin', 'H2O2', 'Spoiled']


## 2. E-Nose Synthetic Data Generation

**Features**: ammonia, alcohol, voc, sulfur, hydrocarbon (5 sensors)

**Range Strategy**: Each adulterant has characteristic sensor signatures (from literature).

In [10]:
def generate_enose_data(n_samples_per_class=100):
    """
    Generate E-Nose synthetic data.
    Returns DataFrame with columns: adulterant, ammonia, alcohol, voc, sulfur, hydrocarbon
    """

    # Sensor ranges and shifts per adulterant (from literature)
    enose_profiles = {
        "Fresh": {
            "ammonia": (0.1, 0.5),
            "alcohol": (0.2, 0.8),
            "voc": (0.3, 0.9),
            "sulfur": (0.05, 0.3),
            "hydrocarbon": (0.1, 0.6)
        },
        "Water": {  # Diluted signals
            "ammonia": (0.05, 0.25),
            "alcohol": (0.1, 0.4),
            "voc": (0.1, 0.5),
            "sulfur": (0.02, 0.15),
            "hydrocarbon": (0.05, 0.3)
        },
        "Urea": {  # High ammonia
            "ammonia": (1.5, 3.0),
            "alcohol": (0.2, 0.8),
            "voc": (0.3, 0.9),
            "sulfur": (0.1, 0.5),
            "hydrocarbon": (0.1, 0.6)
        },
        "Detergent": {  # Variable profile, some alcohol
            "ammonia": (0.2, 0.6),
            "alcohol": (0.5, 1.5),
            "voc": (0.8, 2.0),
            "sulfur": (0.2, 0.8),
            "hydrocarbon": (0.2, 1.0)
        },
        "Formalin": {  # Pungent signals
            "ammonia": (0.5, 1.5),
            "alcohol": (1.5, 3.5),
            "voc": (2.0, 4.5),
            "sulfur": (0.5, 1.5),
            "hydrocarbon": (0.3, 1.0)
        },
        "H2O2": {  # Oxidative markers
            "ammonia": (0.1, 0.4),
            "alcohol": (0.3, 1.0),
            "voc": (0.5, 1.5),
            "sulfur": (0.05, 0.25),
            "hydrocarbon": (0.05, 0.3)
        },
        "Spoiled": {  # Acidic fermentation markers
            "ammonia": (0.3, 1.0),
            "alcohol": (0.5, 2.0),
            "voc": (0.8, 2.5),
            "sulfur": (0.3, 1.0),
            "hydrocarbon": (0.1, 0.8)
        }
    }

    data = []
    for adulterant in ADULTERANTS:
        profile = enose_profiles[adulterant]
        for _ in range(n_samples_per_class):
            sample = {"adulterant": adulterant}
            for sensor, (min_val, max_val) in profile.items():
                # Add Gaussian noise around the range
                base = np.random.uniform(min_val, max_val)
                noise = np.random.normal(0, 0.1 * (max_val - min_val))
                sample[sensor] = max(0, base + noise)
            data.append(sample)

    return pd.DataFrame(data)

print("[OK] E-Nose generation function defined")

[OK] E-Nose generation function defined


In [11]:
# Generate E-Nose data
print("Generating E-Nose synthetic data...")
enose_df = generate_enose_data(n_samples_per_class=1000)
print(f"[OK] Generated {len(enose_df)} E-Nose samples")
print(f"\nE-Nose data sample:")
print(enose_df.head(10))
print(f"\nE-Nose by class:")
print(enose_df["adulterant"].value_counts())

Generating E-Nose synthetic data...
[OK] Generated 7000 E-Nose samples

E-Nose data sample:
  adulterant   ammonia   alcohol       voc    sulfur  hydrocarbon
0      Fresh  0.205341  0.312731  0.351593  0.252281     0.081248
1      Fresh  0.152355  0.225792  0.531760  0.192262     0.246061
2      Fresh  0.257249  0.544404  0.662782  0.079745     0.102513
3      Fresh  0.176966  0.549247  0.587401  0.052861     0.450729
4      Fresh  0.335817  0.382531  0.894869  0.302513     0.093792
5      Fresh  0.145208  0.616442  0.572782  0.099784     0.508891
6      Fresh  0.400222  0.232113  0.727373  0.093381     0.314477
7      Fresh  0.319095  0.670058  0.723399  0.277582     0.363807
8      Fresh  0.293148  0.261083  0.233732  0.203205     0.527062
9      Fresh  0.342418  0.324549  0.371121  0.294961     0.593640

E-Nose by class:
adulterant
Fresh        1000
Water        1000
Urea         1000
Detergent    1000
Formalin     1000
H2O2         1000
Spoiled      1000
Name: count, dtype: int64


## 3. E-Tongue Synthetic Data Generation

**Features**: sweetness, saltiness, sourness, bitterness, umami, astringency (6 sensors)

In [12]:
def generate_etongue_data(n_samples_per_class=100):
    """
    Generate E-Tongue synthetic data.
    Returns DataFrame with columns: adulterant, sweetness, saltiness, sourness, bitterness, umami, astringency
    """

    etongue_profiles = {
        "Fresh": {
            "sweetness": (4.0, 6.0),
            "saltiness": (1.0, 2.5),
            "sourness": (0.5, 1.5),
            "bitterness": (0.1, 0.5),
            "umami": (0.5, 1.5),
            "astringency": (0.2, 0.8)
        },
        "Water": {  # Diluted taste
            "sweetness": (2.0, 3.5),
            "saltiness": (0.3, 1.0),
            "sourness": (0.2, 0.8),
            "bitterness": (0.05, 0.2),
            "umami": (0.1, 0.5),
            "astringency": (0.05, 0.3)
        },
        "Urea": {  # High bitterness
            "sweetness": (2.0, 4.0),
            "saltiness": (1.5, 3.0),
            "sourness": (0.5, 1.5),
            "bitterness": (2.0, 4.0),  # ** MARKER
            "umami": (0.2, 1.0),
            "astringency": (1.0, 2.0)  # ** MARKER
        },
        "Detergent": {  # Harsh, bitter, soapy
            "sweetness": (0.5, 1.5),
            "saltiness": (2.0, 3.5),
            "sourness": (0.3, 1.0),
            "bitterness": (2.5, 4.5),  # ** MARKER
            "umami": (0.1, 0.5),
            "astringency": (2.0, 3.5)  # ** MARKER
        },
        "Formalin": {  # Pungent, sour, harsh
            "sweetness": (0.2, 1.0),
            "saltiness": (1.0, 2.5),
            "sourness": (3.0, 5.0),  # ** MARKER
            "bitterness": (1.5, 3.0),
            "umami": (0.1, 0.5),
            "astringency": (3.0, 4.5)  # ** MARKER
        },
        "H2O2": {  # Clean, faint
            "sweetness": (3.0, 5.0),
            "saltiness": (0.2, 1.0),
            "sourness": (0.1, 0.5),
            "bitterness": (0.05, 0.3),
            "umami": (0.1, 0.5),
            "astringency": (0.1, 0.5)
        },
        "Spoiled": {  # Sour, acidic
            "sweetness": (1.0, 2.5),
            "saltiness": (1.5, 2.5),
            "sourness": (3.5, 5.5),  # ** MARKER
            "bitterness": (0.5, 1.5),
            "umami": (0.1, 0.5),
            "astringency": (1.5, 3.0)  # ** MARKER
        }
    }

    data = []
    for adulterant in ADULTERANTS:
        profile = etongue_profiles[adulterant]
        for _ in range(n_samples_per_class):
            sample = {"adulterant": adulterant}
            for sensor, (min_val, max_val) in profile.items():
                base = np.random.uniform(min_val, max_val)
                noise = np.random.normal(0, 0.15 * (max_val - min_val))
                sample[sensor] = max(0, base + noise)
            data.append(sample)

    return pd.DataFrame(data)

print("[OK] E-Tongue generation function defined")

[OK] E-Tongue generation function defined


In [13]:
# Generate E-Tongue data
print("Generating E-Tongue synthetic data...")
etongue_df = generate_etongue_data(n_samples_per_class=1000)
print(f"[OK] Generated {len(etongue_df)} E-Tongue samples")
print(f"\nE-Tongue data sample:")
print(etongue_df.head(10))
print(f"\nE-Tongue by class:")
print(etongue_df["adulterant"].value_counts())

Generating E-Tongue synthetic data...
[OK] Generated 7000 E-Tongue samples

E-Tongue data sample:
  adulterant  sweetness  saltiness  sourness  bitterness     umami  \
0      Fresh   5.239756   2.372301  0.706173    0.121964  0.764637   
1      Fresh   6.006347   1.418688  1.093750    0.342471  0.521868   
2      Fresh   6.125236   1.272289  0.465732    0.214309  0.683349   
3      Fresh   5.756209   2.294604  1.457749    0.259801  0.540307   
4      Fresh   5.336619   2.387838  0.872347    0.408727  0.704226   
5      Fresh   4.921458   1.416607  1.356537    0.120584  1.111064   
6      Fresh   3.739093   1.997609  1.123393    0.460281  0.614048   
7      Fresh   3.967556   1.296639  0.877499    0.196375  0.908602   
8      Fresh   5.636474   1.150595  1.048236    0.236977  0.956796   
9      Fresh   4.803759   1.699637  1.100362    0.129279  0.604843   

   astringency  
0     0.292168  
1     0.445908  
2     0.588943  
3     0.582916  
4     0.470816  
5     0.549719  
6     0.2008

In [14]:
# Generate vision images
print("Generating vision deposit pattern images...")
vision_results = generate_vision_images(n_samples_per_class=60)
print(f"[OK] Generated {len(vision_results)} vision images")
print(f"\nSample vision results (first 5):")
for img_path, deposit_type, adulterant in vision_results[:5]:
    print(f"  {adulterant}: {img_path} [{deposit_type}]")

Generating vision deposit pattern images...
[OK] Generated 420 vision images

Sample vision results (first 5):
  Fresh: ../data/synthetic/vision/Fresh/fresh_0000.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0001.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0002.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0003.png [clean_ring]
  Fresh: ../data/synthetic/vision/Fresh/fresh_0004.png [clean_ring]


In [15]:
# Insert E-Nose data into DB
print("Inserting E-Nose data into DB...")
aroma_count = 0
for idx, row in enose_df.iterrows():
    # First, create a batch for this sample
    batch_id = insert_batch(adulterant=row["adulterant"], db_path=DB_PATH)
    pred_class, confidence = mock_predict_class(row["adulterant"])

    try:
        insert_aroma_analysis(
            batch_id=batch_id,
            ammonia=float(row["ammonia"]),
            alcohol=float(row["alcohol"]),
            voc=float(row["voc"]),
            sulfur=float(row["sulfur"]),
            hydrocarbon=float(row["hydrocarbon"]),
            predicted_class=pred_class,
            confidence=confidence,
            db_path=DB_PATH
        )
        aroma_count += 1
    except Exception as e:
        print(f"[ERROR] Row {idx}: {e}")
        break

    if (idx + 1) % 1000 == 0:
        print(f"  ...inserted {idx + 1}/{len(enose_df)} E-Nose samples")

print(f"[OK] Inserted {aroma_count} E-Nose samples into DB")

Inserting E-Nose data into DB...
  ...inserted 1000/7000 E-Nose samples
  ...inserted 2000/7000 E-Nose samples
  ...inserted 3000/7000 E-Nose samples
  ...inserted 4000/7000 E-Nose samples
  ...inserted 5000/7000 E-Nose samples
  ...inserted 6000/7000 E-Nose samples
  ...inserted 7000/7000 E-Nose samples
[OK] Inserted 7000 E-Nose samples into DB


In [16]:
# Insert E-Tongue data into DB
print("Inserting E-Tongue data into DB...")
taste_count = 0
for idx, row in etongue_df.iterrows():
    # First, create a batch for this sample
    batch_id = insert_batch(adulterant=row["adulterant"], db_path=DB_PATH)
    pred_class, confidence = mock_predict_class(row["adulterant"])

    try:
        insert_taste_analysis(
            batch_id=batch_id,
            sweetness=float(row["sweetness"]),
            saltiness=float(row["saltiness"]),
            sourness=float(row["sourness"]),
            bitterness=float(row["bitterness"]),
            umami=float(row["umami"]),
            astringency=float(row["astringency"]),
            predicted_class=pred_class,
            confidence=confidence,
            db_path=DB_PATH
        )
        taste_count += 1
    except Exception as e:
        print(f"[ERROR] Row {idx}: {e}")
        break

    if (idx + 1) % 1000 == 0:
        print(f"  ...inserted {idx + 1}/{len(etongue_df)} E-Tongue samples")

print(f"[OK] Inserted {taste_count} E-Tongue samples into DB")

Inserting E-Tongue data into DB...
  ...inserted 1000/7000 E-Tongue samples
  ...inserted 2000/7000 E-Tongue samples
  ...inserted 3000/7000 E-Tongue samples
  ...inserted 4000/7000 E-Tongue samples
  ...inserted 5000/7000 E-Tongue samples
  ...inserted 6000/7000 E-Tongue samples
  ...inserted 7000/7000 E-Tongue samples
[OK] Inserted 7000 E-Tongue samples into DB


In [17]:
# Insert Vision data into DB
print("Inserting Vision data into DB...")
vision_count = 0
for img_path, deposit_type, adulterant in vision_results:
    # First, create a batch for this sample
    batch_id = insert_batch(adulterant=adulterant, db_path=DB_PATH)
    pred_class, confidence = mock_predict_class(adulterant)

    try:
        insert_vision_analysis(
            batch_id=batch_id,
            image_path=img_path,
            deposit_type=deposit_type,
            predicted_class=pred_class,
            confidence=confidence,
            db_path=DB_PATH
        )
        vision_count += 1
    except Exception as e:
        print(f"[ERROR] {img_path}: {e}")
        break

    if (vision_count) % 100 == 0:
        print(f"  ...inserted {vision_count}/{len(vision_results)} Vision samples")

print(f"[OK] Inserted {vision_count} Vision samples into DB")

Inserting Vision data into DB...
[ERROR] ../data/synthetic/vision/Fresh/fresh_0000.png: insert_vision_analysis() missing 2 required positional arguments: 'sample_id' and 'object_detection_info'
[OK] Inserted 0 Vision samples into DB


## 7. Verify Data in DB

In [18]:
# Check aroma_analysis table
aroma_query = "SELECT COUNT(*) as count FROM aroma_analysis"
aroma_rows = execute_query(DB_PATH, aroma_query)
print(f"Total aroma_analysis records: {dict(aroma_rows[0])['count']}")

# Check taste_analysis table
taste_query = "SELECT COUNT(*) as count FROM taste_analysis"
taste_rows = execute_query(DB_PATH, taste_query)
print(f"Total taste_analysis records: {dict(taste_rows[0])['count']}")

# Check vision_analysis table
vision_query = "SELECT COUNT(*) as count FROM vision_analysis"
vision_rows = execute_query(DB_PATH, vision_query)
print(f"Total vision_analysis records: {dict(vision_rows[0])['count']}")

Total aroma_analysis records: 14000
Total taste_analysis records: 14000
Total vision_analysis records: 0


## 8. Sample Data Inspection

In [19]:
# Show sample aroma record
query = "SELECT * FROM aroma_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Aroma Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

print("\n" + "="*50 + "\n")

# Show sample taste record
query = "SELECT * FROM taste_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Taste Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

print("\n" + "="*50 + "\n")

# Show sample vision record
query = "SELECT * FROM vision_analysis LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    print("Sample Vision Record:")
    for key, value in dict(rows[0]).items():
        print(f"  {key}: {value}")

Sample Aroma Record:
  batch_id: BATCH-64AFD60E
  ground_truth: None
  ammonia: 0.2053408428170682
  alcohol: 0.31273084328308465
  voc: 0.3515926448329279
  sulfur: 0.2522810265691745
  hydrocarbon: 0.08124834044672548
  predicted_class: Fresh
  confidence: 0.95
  created_at: 2026-06-15 17:34:58


Sample Taste Record:
  batch_id: BATCH-DE1CF1FE
  ground_truth: None
  sweetness: 5.239755821423183
  saltiness: 2.372300670166239
  sourness: 0.7061726107133135
  bitterness: 0.12196356462089898
  umami: 0.7646366317697724
  astringency: 0.29216758197571485
  predicted_class: Fresh
  confidence: 0.95
  created_at: 2026-06-15 17:37:58


